# Wildfire Field Lab · CPU and CUDA benchmark

Choose **Runtime → Change runtime type → T4 GPU** (or another CUDA GPU), then **Run all**. Upload `Andrew_Wildfire_Hackathon_Starter.zip` or its extracted `norcal_next_day_replay.csv.gz` when prompted.

One runtime measures the direct CPU calculation, the tiled CPU calculation, then the same tiles on CUDA over 127,650,144 historical comparisons. The benchmark checks scores and top-ten sets. It reports CUDA kernel time and time including transfers separately. You can load the downloaded result in the Field Lab benchmark panel.

In [ ]:
import json
import os
from pathlib import Path
import subprocess
import sys
import tempfile

import torch
from google.colab import files

if not torch.cuda.is_available():
    raise RuntimeError("Choose a CUDA GPU under Runtime → Change runtime type, then rerun.")
print("CUDA device", torch.cuda.get_device_name(0))

workspace = Path(tempfile.mkdtemp(prefix="wildfire-benchmark-", dir="/content"))
repository = workspace / "source"
subprocess.run([
    "git", "clone", "--quiet", "--depth", "1",
    "https://github.com/ahackv/wIldfire-predictor.git", str(repository),
], check=True)
revision = subprocess.check_output(["git", "-C", str(repository), "rev-parse", "HEAD"], text=True)
print("Benchmark source", revision.strip())

In [ ]:
uploads = files.upload()
supported = {"Andrew_Wildfire_Hackathon_Starter.zip", "norcal_next_day_replay.csv.gz"}
matches = [name for name in uploads if name in supported]
if len(matches) != 1:
    raise ValueError("Upload the starter ZIP or the extracted replay CSV, one file at a time.")
replay_path = workspace / matches[0]
replay_path.write_bytes(uploads[matches[0]])
del uploads

In [ ]:
report_path = workspace / "wildfire-benchmark.json"
workers = max(1, min(4, len(os.sched_getaffinity(0))))
subprocess.run([
    sys.executable, str(repository / "scripts" / "benchmark_parallel.py"), str(replay_path),
    "--device", "cuda", "--workers", str(workers), "--repeats", "5",
    "--output", str(report_path),
], cwd=repository, check=True)
report = json.loads(report_path.read_text())
gpu = report["cuda"]
print(f"Fastest CPU {min(report['referenceSeconds'], report['batchedSeconds'], report['parallelSeconds']):.4f} s")
print(f"CUDA kernel {gpu['kernelSeconds']:.4f} s · {gpu['kernelSpeedup']:.2f}× fastest CPU")
print(f"CUDA with transfers {gpu['transferInclusiveSeconds']:.4f} s · {gpu['transferInclusiveSpeedup']:.2f}× fastest CPU")
print(f"Top-ten sets agreed on {gpu['topTenAgreementDays']} of {report['days']} days")
files.download(str(report_path))